# STRATA Cross-Model Benchmark on Google Colab (T4 GPU)

Evaluates:
- **Qwen3.8-27B** (`unsloth/Qwen3.8-27B-GGUF`, UD-IQ3_S, 12.0 GB)
- **GPT-OSS 20B** (`unsloth/gpt-oss-20b-GGUF`, Q4_K_M, 11.6 GB)
- **Qwen3-Coder-30B-A3B** (`unsloth/Qwen3-Coder-30B-A3B-Instruct-GGUF`, Q3_K_S, 13.3 GB)
- **Qwen3.5 Highest T4 Fit**: `qwen3.5-35b-a3b` (35B MoE UD-IQ3_XXS, 13.1 GB) and `qwen3.5-27b` (27B Dense Q3_K_S, 12.3 GB)

Uses latest prebuilt `llama-server` (CUDA 12.8) from `ggml-org/llama.cpp` so all 2026 architectures run immediately on T4 without Python wheel compilation lag.

In [ ]:
# Cell 1: Verify Colab T4 GPU & clone STRATA repo (dev branch)
!nvidia-smi
%cd /content
!rm -rf STRATA
!git clone -b dev https://github.com/MatthewNader2/STRATA.git
%cd /content/STRATA

In [ ]:
# Cell 2: Install delta Python deps + latest prebuilt CUDA llama-server binary
!pip -q install -r benchmarks/requirements-colab.txt

import json, urllib.request, tarfile, pathlib, os
req = urllib.request.Request('https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=5', headers={'User-Agent': 'Mozilla/5.0'})
releases = json.loads(urllib.request.urlopen(req, timeout=15).read())
llama_dir = pathlib.Path('/content/llama_bin')
llama_dir.mkdir(parents=True, exist_ok=True)
for rel in releases:
    cuda_assets = [a for a in rel['assets'] if 'bin-ubuntu-cuda-12.8-x64.tar.gz' in a['name']]
    if cuda_assets:
        for asset in cuda_assets:
            dst = pathlib.Path('/content') / asset['name']
            if not dst.exists():
                print('Downloading', asset['name'])
                urllib.request.urlretrieve(asset['browser_download_url'], dst)
            !tar -xzf {dst} -C /content/llama_bin
        break

server_bins = list(llama_dir.rglob('llama-server'))
assert server_bins, 'llama-server binary not found!'
LLAMA_SERVER = str(server_bins[0])
print('LLAMA_SERVER =', LLAMA_SERVER)

In [ ]:
# Cell 3: Run Core 9 Suites (8 test.py presets + 3 CryoPulse unseen-domain tasks)
# Uses --cleanup-cache so each ~12-13GB GGUF is deleted right after scoring to fit Colab disk limits.
!python benchmarks/colab_eval.py \
  --llama-server "{LLAMA_SERVER}" \
  --suite core9 \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b qwen3.5-27b \
  --out-dir test_logs/colab_bench \
  --cleanup-cache \
  --resume

In [ ]:
# Cell 4 (Optional): Run Tier-2 Stress Suite (Aetheris + Realistic User + Honest Boundary)
!python benchmarks/colab_eval.py \
  --llama-server "{LLAMA_SERVER}" \
  --suite stress \
  --models qwen3.8-27b gpt-oss-20b qwen3-coder-30b-a3b qwen3.5-35b-a3b \
  --out-dir test_logs/colab_bench_stress \
  --cleanup-cache \
  --resume

In [ ]:
# Cell 5: Generate summary tables & PNG graphs (optionally upload gemma4_responses.txt first)
# If you uploaded gemma4_responses.txt from AI Studio:
import os
if os.path.exists('gemma4_responses.txt'):
    !python benchmarks/ingest_aistudio.py --input gemma4_responses.txt --suite all --out test_logs/colab_bench/results_gemma4_31b.json

!python benchmarks/visualize_results.py test_logs/colab_bench --out-dir test_logs/visualizations
!zip -r /content/strata_colab_results.zip test_logs/colab_bench test_logs/visualizations